# ATC Drug Classification — WHO Therapeutic Class Mapping

**Purpose:** Map all drugs from our FAERS DDI signal analysis to WHO ATC (Anatomical Therapeutic Chemical) classification codes using the RxClass API.

**Why:** Replaces the manual high-risk drug class dictionary with a standardized, reproducible classification. Improves Gephi visualization coloring.

**Input:** `FAERS_DDI_SIGNALS_HIGH_CONFIDENCE.csv` (drug pairs from the main analysis)  
**Output:** `drug_atc_mapping.csv` (drug -> ATC Level 1/2/3 mapping for all unique drugs)

**API Source:** NIH RxClass API (https://rxnav.nlm.nih.gov/REST/rxclass/)

**ATC Hierarchy:**
- Level 1: Anatomical main group (e.g., B = Blood)
- Level 2: Therapeutic subgroup (e.g., B01 = Antithrombotic agents) **primary grouping**
- Level 3: Pharmacological subgroup (e.g., B01A = Antithrombotic agents)
- Level 4: Chemical subgroup (e.g., B01AA = Vitamin K antagonists)
- Level 5: Chemical substance (e.g., B01AA03 = Warfarin)


In [15]:
import pandas as pd
import numpy as np
import requests
import json
import time
import os
from collections import Counter

print("Libraries loaded")


Libraries loaded


## Step 1: Extract Unique Drugs from Signal Data

In [16]:
# ---- LOAD SIGNAL DATA ----
try:
    signals = pd.read_csv("FAERS_DDI_SIGNALS_HIGH_CONFIDENCE.csv")
    print(f"Loaded high-confidence signals: {len(signals):,}")
except FileNotFoundError:
    signals = pd.read_csv("FAERS_DDI_SIGNALS.csv")
    print(f"Loaded all signals: {len(signals):,}")

# ---- EXTRACT UNIQUE DRUG NAMES ----
all_drugs = set()
for pair in signals['PAIR']:
    parts = pair.split(' + ')
    if len(parts) == 2:
        all_drugs.add(parts[0].strip())
        all_drugs.add(parts[1].strip())

all_drugs = sorted(all_drugs)
print(f"Unique drugs to classify: {len(all_drugs)}")
print(f"\nFirst 20 drugs: {all_drugs[:20]}")


Loaded high-confidence signals: 5,501
Unique drugs to classify: 682

First 20 drugs: ['(-)-ARTERENOL', 'ACETAMINOPHEN', 'ACETAMINOPHEN / CODEINE', 'ACETAMINOPHEN / HYDROCODONE', 'ACETAMINOPHEN\\CODEINE PHOSPHATE', 'ACETAMINOPHEN\\OXYCODONE HYDROCHLORIDE', 'ACETYLCYSTEINE', 'ACTEMRA', 'ACUPAN', 'ACYCLOVIR', 'ADALIMUMAB', 'ADCIRCA', 'ADEMPAS', 'ADVAIR HFA', 'AIMOVIG', 'AKYNZEO', 'ALBUMIN HUMAN', 'ALBUTEROL', 'ALBUTEROL / IPRATROPIUM', 'ALBUTEROL SULFATE']


## Step 2: Query RxClass API for ATC Codes

The RxClass API maps drug names to ATC classifications. We query each drug once and cache the results to avoid redundant API calls.

**API endpoint:** `https://rxnav.nlm.nih.gov/REST/rxclass/class/byDrugName.json`  
**Rate limited:**


In [17]:
# ---- RXCLASS API FUNCTION ----
RXCLASS_BASE = "https://rxnav.nlm.nih.gov/REST/rxclass/class/byDrugName.json"
CACHE_FILE = "atc_mapping_cache.json"

def load_cache():
    """Load cached API results to avoid re-querying."""
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, 'r') as f:
            cache = json.load(f)
        print(f"Loaded cache with {len(cache)} entries")
        return cache
    return {}

def save_cache(cache):
    """Save API results."""
    with open(CACHE_FILE, 'w') as f:
        json.dump(cache, f, indent=2)

def get_atc_codes(drug_name, cache):
    """
    Query RxClass API for ATC classifications of a drug.
    Returns list of dicts with ATC code, class name, and level info.
    """
    # Check cache first
    if drug_name in cache:
        return cache[drug_name]

    try:
        params = {
            'drugName': drug_name,
            'relaSource': 'ATC'
        }
        response = requests.get(RXCLASS_BASE, params=params, timeout=10)

        if response.status_code != 200:
            cache[drug_name] = []
            return []

        data = response.json()

        # Parse the response
        atc_entries = []
        if 'rxclassDrugInfoList' in data and 'rxclassDrugInfo' in data['rxclassDrugInfoList']:
            for info in data['rxclassDrugInfoList']['rxclassDrugInfo']:
                class_info = info.get('rxclassMinConceptItem', {})
                atc_code = class_info.get('classId', '')
                class_name = class_info.get('className', '')

                if atc_code:  # Only keep entries with actual ATC codes
                    atc_entries.append({
                        'atc_code': atc_code,
                        'class_name': class_name,
                        'atc_level': len(atc_code)  # 1=L1, 3=L2, 4=L3, 5=L4, 7=L5
                    })

        cache[drug_name] = atc_entries
        return atc_entries

    except Exception as e:
        print(f"  Error for {drug_name}: {e}")
        cache[drug_name] = []
        return []

print("API function defined")
print(f"Endpoint: {RXCLASS_BASE}")


API function defined
Endpoint: https://rxnav.nlm.nih.gov/REST/rxclass/class/byDrugName.json


## Step 3: Query All Drugs

In [18]:
# ---- QUERY ALL DRUGS ----
cache = load_cache()
already_cached = len(cache)

results = {}
failed = []
new_queries = 0

print(f"Starting ATC mapping for {len(all_drugs)} drugs...")
print(f"Already cached: {already_cached}")
print(f"Need to query: {len(all_drugs) - sum(1 for d in all_drugs if d in cache)}")
print("=" * 60)

for i, drug in enumerate(all_drugs):
    atc_entries = get_atc_codes(drug, cache)
    results[drug] = atc_entries

    if drug not in cache or i >= already_cached:
        new_queries += 1
        # Small delay to be polite to the API
        if new_queries % 5 == 0:
            time.sleep(0.3)

    # Progress update every 50 drugs
    if (i + 1) % 50 == 0:
        print(f"  Processed {i+1}/{len(all_drugs)} drugs ({new_queries} new API calls)")
        save_cache(cache)  # Save periodically

    if not atc_entries:
        failed.append(drug)

# Final save
save_cache(cache)

print(f"\n{'=' * 60}")
print(f"COMPLETE")
print(f"  Total drugs: {len(all_drugs)}")
print(f"  Successfully mapped: {len(all_drugs) - len(failed)}")
print(f"  No ATC code found: {len(failed)}")
print(f"  New API calls made: {new_queries}")
print(f"  Cache saved to: {CACHE_FILE}")


Loaded cache with 812 entries
Starting ATC mapping for 682 drugs...
Already cached: 812
Need to query: 1
  Processed 50/682 drugs (0 new API calls)
  Processed 100/682 drugs (0 new API calls)
  Processed 150/682 drugs (0 new API calls)
  Processed 200/682 drugs (0 new API calls)
  Processed 250/682 drugs (0 new API calls)
  Processed 300/682 drugs (0 new API calls)
  Processed 350/682 drugs (0 new API calls)
  Processed 400/682 drugs (0 new API calls)
  Processed 450/682 drugs (0 new API calls)
  Processed 500/682 drugs (0 new API calls)
  Processed 550/682 drugs (0 new API calls)
  Processed 600/682 drugs (0 new API calls)
  Processed 650/682 drugs (0 new API calls)

COMPLETE
  Total drugs: 682
  Successfully mapped: 565
  No ATC code found: 117
  New API calls made: 0
  Cache saved to: atc_mapping_cache.json


## Step 4: Build the ATC Mapping Table

For each drug, we extract the most useful ATC level:
- **Level 2** (3 chars, e.g., "B01"): Primary grouping for coloring and analysis
- **Level 1** (1 char, e.g., "B"): Broad anatomical category

Some drugs map to multiple ATC codes (e.g., aspirin is both an analgesic and an antiplatelet). We keep the **first** classification returned, which is typically the primary therapeutic use.


In [19]:
# ---- BUILD MAPPING TABLE ----
mapping_rows = []

# ATC Level 1 descriptions (for readable labels)
ATC_L1_NAMES = {
    'A': 'Alimentary/Metabolism',
    'B': 'Blood',
    'C': 'Cardiovascular',
    'D': 'Dermatologicals',
    'G': 'Genitourinary/Sex Hormones',
    'H': 'Systemic Hormones',
    'J': 'Anti-infectives',
    'L': 'Antineoplastic/Immunomodulating',
    'M': 'Musculoskeletal',
    'N': 'Nervous System',
    'P': 'Antiparasitic',
    'R': 'Respiratory',
    'S': 'Sensory Organs',
    'V': 'Various'
}

for drug in all_drugs:
    entries = results.get(drug, [])

    if not entries:
        mapping_rows.append({
            'DRUG': drug,
            'ATC_CODE_FULL': '',
            'ATC_L1_CODE': '',
            'ATC_L1_NAME': 'UNCLASSIFIED',
            'ATC_L2_CODE': '',
            'ATC_L2_NAME': 'UNCLASSIFIED',
            'ATC_L3_CODE': '',
            'ATC_L3_NAME': '',
            'NUM_ATC_CLASSES': 0
        })
        continue

    # Get entries by level
    l1_entries = [e for e in entries if e['atc_level'] == 1]
    l2_entries = [e for e in entries if e['atc_level'] == 3]  # 3 chars = Level 2
    l3_entries = [e for e in entries if e['atc_level'] == 4]  # 4 chars = Level 3
    l4_entries = [e for e in entries if e['atc_level'] == 5]  # 5 chars = Level 4

    # Use the most specific available, prefer L2 for grouping
    best_l1 = l1_entries[0] if l1_entries else None
    best_l2 = l2_entries[0] if l2_entries else None
    best_l3 = l3_entries[0] if l3_entries else None

    # If no L2 but we have other levels, derive L1 from the code
    atc_l1_code = ''
    atc_l1_name = 'UNCLASSIFIED'
    if best_l1:
        atc_l1_code = best_l1['atc_code']
        atc_l1_name = ATC_L1_NAMES.get(atc_l1_code, best_l1['class_name'])
    elif best_l2:
        atc_l1_code = best_l2['atc_code'][0]
        atc_l1_name = ATC_L1_NAMES.get(atc_l1_code, '')
    elif best_l3:
        atc_l1_code = best_l3['atc_code'][0]
        atc_l1_name = ATC_L1_NAMES.get(atc_l1_code, '')
    elif entries:
        # Fall back to first entry
        code = entries[0]['atc_code']
        if len(code) >= 1:
            atc_l1_code = code[0]
            atc_l1_name = ATC_L1_NAMES.get(atc_l1_code, '')

    mapping_rows.append({
        'DRUG': drug,
        'ATC_CODE_FULL': entries[0]['atc_code'] if entries else '',
        'ATC_L1_CODE': atc_l1_code,
        'ATC_L1_NAME': atc_l1_name,
        'ATC_L2_CODE': best_l2['atc_code'] if best_l2 else '',
        'ATC_L2_NAME': best_l2['class_name'] if best_l2 else '',
        'ATC_L3_CODE': best_l3['atc_code'] if best_l3 else '',
        'ATC_L3_NAME': best_l3['class_name'] if best_l3 else '',
        'NUM_ATC_CLASSES': len(set(e['atc_code'] for e in entries))
    })

atc_df = pd.DataFrame(mapping_rows)
print(f"Mapping table created: {len(atc_df)} drugs")
print(f"\nClassification coverage:")
print(f"  Drugs with ATC code:    {(atc_df['ATC_L1_CODE'] != '').sum()}")
print(f"  Drugs unclassified:     {(atc_df['ATC_L1_CODE'] == '').sum()}")
print(f"  Coverage rate:          {(atc_df['ATC_L1_CODE'] != '').sum() / len(atc_df) * 100:.1f}%")


Mapping table created: 682 drugs

Classification coverage:
  Drugs with ATC code:    565
  Drugs unclassified:     117
  Coverage rate:          82.8%


In [20]:
# ---- FIX: DERIVE ATC LEVELS BY SLICING ----
print("Re-deriving ATC levels from raw codes...")

for idx, row in atc_df.iterrows():
    drug = row['DRUG']
    entries = results.get(drug, [])

    if not entries:
        continue

    # Get the first (primary) ATC code returned
    primary_code = entries[0]['atc_code']

    # Derive levels by slicing
    if len(primary_code) >= 1:
        atc_df.at[idx, 'ATC_L1_CODE'] = primary_code[0]
        atc_df.at[idx, 'ATC_L1_NAME'] = ATC_L1_NAMES.get(primary_code[0], '')
    if len(primary_code) >= 3:
        atc_df.at[idx, 'ATC_L2_CODE'] = primary_code[:3]
        atc_df.at[idx, 'ATC_L2_NAME'] = entries[0]['class_name']  # Use API name
    if len(primary_code) >= 4:
        atc_df.at[idx, 'ATC_L3_CODE'] = primary_code[:4]
        atc_df.at[idx, 'ATC_L3_NAME'] = entries[0]['class_name']

# Verify it worked
print(f"\nDrugs with L2 code now: {(atc_df['ATC_L2_CODE'] != '').sum()}")
print(f"\nSample mappings:")
test = atc_df[atc_df['DRUG'].isin(['METHOTREXATE','ASPIRIN','MORPHINE','ADALIMUMAB'])]
print(test[['DRUG','ATC_L1_NAME','ATC_L2_CODE']].to_string(index=False))

Re-deriving ATC levels from raw codes...

Drugs with L2 code now: 565

Sample mappings:
        DRUG                     ATC_L1_NAME ATC_L2_CODE
  ADALIMUMAB Antineoplastic/Immunomodulating         L04
     ASPIRIN           Alimentary/Metabolism         A01
METHOTREXATE Antineoplastic/Immunomodulating         L01
    MORPHINE                  Nervous System         N02


## Step 5: Review the Classification Distribution

In [21]:
# ---- LEVEL 1 DISTRIBUTION ----
print("ATC LEVEL 1 — Anatomical Main Groups")
print("=" * 60)
l1_counts = atc_df[atc_df['ATC_L1_NAME'] != 'UNCLASSIFIED']['ATC_L1_NAME'].value_counts()
for name, count in l1_counts.items():
    print(f"  {name:40s} {count:4d}")
print(f"  {'UNCLASSIFIED':40s} {(atc_df['ATC_L1_NAME'] == 'UNCLASSIFIED').sum():4d}")

print(f"\n\nATC LEVEL 2 — Therapeutic Subgroups (top 20)")
print("=" * 60)
l2_counts = atc_df[atc_df['ATC_L2_NAME'] != '']['ATC_L2_NAME'].value_counts().head(20)
for name, count in l2_counts.items():
    print(f"  {name:50s} {count:4d}")


ATC LEVEL 1 — Anatomical Main Groups
  Nervous System                            114
  Alimentary/Metabolism                     106
  Antineoplastic/Immunomodulating           101
  Cardiovascular                             72
  Anti-infectives                            38
  Respiratory                                33
  Blood                                      30
  Dermatologicals                            25
  Musculoskeletal                            22
  Genitourinary/Sex Hormones                 10
  Antiparasitic                               4
  Systemic Hormones                           4
  Various                                     3
  Sensory Organs                              3
  UNCLASSIFIED                              117


ATC LEVEL 2 — Therapeutic Subgroups (top 20)
  UNCLASSIFIED                                        117
  Tumor necrosis factor alpha (TNF-alpha) inhibitors   11
  Benzodiazepine derivatives                           10
  Platelet aggregation

## Step 6: Handle Unclassified Drugs

Some drugs won't map via the API — typically because:
- They're combination products (e.g., ACETAMINOPHEN/CODEINE)
- They're international name variants (e.g., DICLOFENACO)
- They're supplements or non-prescription items (e.g., CALCIUM CARBONATE)

For combinations, we can try mapping each component separately and take the most clinically relevant class.


In [22]:
# ---- LIST UNCLASSIFIED DRUGS ----
unclassified = atc_df[atc_df['ATC_L1_CODE'] == '']['DRUG'].tolist()
print(f"Unclassified drugs ({len(unclassified)}):")
for drug in sorted(unclassified):
    print(f"  {drug}")

# ---- TRY COMBINATION DRUG COMPONENTS ----
print(f"\n{'=' * 60}")
print("Attempting to classify combination products by components...")

reclassified = 0
for idx, row in atc_df[atc_df['ATC_L1_CODE'] == ''].iterrows():
    drug = row['DRUG']

    # Try splitting on common separators: /, ;, AND
    components = []
    if '/' in drug:
        components = [c.strip() for c in drug.split('/')]
    elif ';' in drug:
        components = [c.strip() for c in drug.split(';')]

    if not components:
        continue

    # Try each component
    for comp in components:
        entries = get_atc_codes(comp, cache)
        if entries:
            l2_entries = [e for e in entries if e['atc_level'] == 3]
            l1_entries = [e for e in entries if e['atc_level'] == 1]

            if l2_entries:
                atc_df.at[idx, 'ATC_L2_CODE'] = l2_entries[0]['atc_code']
                atc_df.at[idx, 'ATC_L2_NAME'] = l2_entries[0]['class_name']
            if l1_entries:
                atc_df.at[idx, 'ATC_L1_CODE'] = l1_entries[0]['atc_code']
                atc_df.at[idx, 'ATC_L1_NAME'] = ATC_L1_NAMES.get(l1_entries[0]['atc_code'], l1_entries[0]['class_name'])
            elif l2_entries:
                atc_df.at[idx, 'ATC_L1_CODE'] = l2_entries[0]['atc_code'][0]
                atc_df.at[idx, 'ATC_L1_NAME'] = ATC_L1_NAMES.get(l2_entries[0]['atc_code'][0], '')

            atc_df.at[idx, 'ATC_CODE_FULL'] = entries[0]['atc_code']
            reclassified += 1
            print(f"  {drug} -> {comp} -> {atc_df.at[idx, 'ATC_L1_NAME']}")
            break  # Use first successful component

save_cache(cache)
print(f"\nReclassified {reclassified} combination products")
print(f"\nUpdated coverage:")
print(f"  Drugs with ATC code:    {(atc_df['ATC_L1_CODE'] != '').sum()}")
print(f"  Still unclassified:     {(atc_df['ATC_L1_CODE'] == '').sum()}")
print(f"  Coverage rate:          {(atc_df['ATC_L1_CODE'] != '').sum() / len(atc_df) * 100:.1f}%")


Unclassified drugs (117):
  (-)-ARTERENOL
  ACETAMINOPHEN\CODEINE PHOSPHATE
  ACETAMINOPHEN\OXYCODONE HYDROCHLORIDE
  ACUPAN
  ADVAIR HFA
  ALBUMIN HUMAN
  ALCOHOL
  ALENDRONATE SODIUM
  ANORO ELLIPTA
  ANTI-THYMOCYTE GLOBULIN
  AZILSARTAN KAMEDOXOMIL
  AZOSEMIDE
  AZTREONAM LYSINE
  BACTRIM DS
  BENZOYLECGONINE
  BMS-830216
  BOTULINUM TOXIN TYPE A
  BREO ELLIPTA
  BUDESONIDE\FORMOTEROL FUMARATE
  CALCIUM
  CALONAL
  CANNABIS SATIVA SUBSP. INDICA TOP
  CAVIAR, UNSPECIFIED
  CELLCEPT
  CETIRIZINE HYDROCHLORIDE\PSEUDOEPHEDRINE HYDROCHLORIDE
  CICLOBENZAPRINA
  CLAVULANIC ACID
  CLORHEXIDINA
  COCODAMOL
  COVID-19 VACCINE MRNA
  CYANOCOBALAMIN
  DARATUMUMAB/HYALURONIDASE-FIHJ
  DELFLEX WITH DEXTROSE 15% LOW MAGNESIUM LOW CALCIUM
  DELFLEX WITH DEXTROSE 25% LOW MAGNESIUM LOW CALCIUM
  DELFLEX WITH DEXTROSE 425% LOW MAGNESIUM LOW CALCIUM
  DEVICE
  DEXTROSE
  DIANEAL PD-2 PERITONEAL DIALYSIS SOLUTION WITH DEXTROSE
  DIANEAL PD-2 WITH DEXTROSE
  DIVALPROEX SODIUM
  DONEPEZILO
  DOXAZOSINA
 

In [23]:
# =============================================================================
# FIX 1: SYSTEMIC-USE PRIORITY FOR MULTI-CODE DRUGS
# =============================================================================
# For DDI research, systemic therapeutic use matters more than local/topical.
# Rank ATC Level 2 prefixes by relevance to drug-drug interactions.

# Higher number = more relevant for DDI analysis
ATC_L2_PRIORITY = {
    # HIGH PRIORITY — systemic drugs most relevant to DDIs
    'B01': 90,  # Antithrombotics (warfarin, aspirin antiplatelet use)
    'L01': 90,  # Antineoplastics
    'L04': 90,  # Immunosuppressants
    'N02': 85,  # Analgesics/opioids
    'M01': 85,  # NSAIDs (systemic)
    'J01': 85,  # Antibacterials (systemic)
    'J02': 85,  # Antifungals (systemic)
    'J05': 85,  # Antivirals
    'H02': 85,  # Systemic corticosteroids
    'N05': 80,  # Psycholeptics (antipsychotics, anxiolytics)
    'N06': 80,  # Psychoanaleptics (antidepressants)
    'N03': 80,  # Antiepileptics
    'C09': 80,  # RAAS inhibitors
    'C07': 80,  # Beta blockers
    'C10': 80,  # Lipid modifying
    'C01': 80,  # Cardiac therapy
    'C08': 80,  # Calcium channel blockers
    'A10': 80,  # Antidiabetics
    'A02': 75,  # Antacids/PPIs
    'R03': 75,  # Anti-asthmatics
    'N01': 70,  # Anesthetics
    'N07': 70,  # Other nervous system
    'L03': 70,  # Immunostimulants
    'L02': 70,  # Endocrine therapy
    'B02': 70,  # Antihemorrhagics
    'B03': 70,  # Antianemia
    'C03': 70,  # Diuretics
    'C02': 70,  # Antihypertensives
    'G04': 65,  # Urologicals
    'A07': 65,  # Antidiarrheals
    'P01': 65,  # Antiprotozoals

    # LOW PRIORITY — topical/local, less relevant to systemic DDIs
    'A01': 20,  # Stomatological (local oral)
    'D07': 20,  # Dermatological corticosteroids
    'D10': 20,  # Anti-acne
    'D04': 20,  # Antipruritics
    'D06': 20,  # Antibiotics for dermatological use
    'D08': 15,  # Antiseptics
    'D09': 15,  # Medicated dressings
    'D11': 15,  # Other dermatological
    'S01': 20,  # Ophthalmologicals
    'S02': 20,  # Otologicals
    'S03': 20,  # Ophth/otological combo
    'C05': 20,  # Vasoprotectives (topical)
    'R01': 25,  # Nasal preparations
    'R02': 25,  # Throat preparations
    'M02': 25,  # Topical joint/muscle
    'B05': 25,  # Blood substitutes/perfusion
    'G01': 25,  # Gynaecological anti-infectives
}

def pick_best_atc(entries):
    """Pick the most DDI-relevant ATC code from multiple entries."""
    if not entries:
        return None
    if len(entries) == 1:
        return entries[0]

    # Score each entry by L2 priority
    scored = []
    for entry in entries:
        code = entry['atc_code']
        l2 = code[:3] if len(code) >= 3 else ''
        priority = ATC_L2_PRIORITY.get(l2, 50)  # Default 50 for unknown
        scored.append((priority, entry))

    # Return highest priority
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1]

# Re-derive all mappings using priority selection
rederived = 0
for idx, row in atc_df.iterrows():
    drug = row['DRUG']
    entries = results.get(drug, [])

    if not entries:
        continue

    best = pick_best_atc(entries)
    code = best['atc_code']

    old_l2 = row['ATC_L2_CODE']

    if len(code) >= 1:
        atc_df.at[idx, 'ATC_L1_CODE'] = code[0]
        atc_df.at[idx, 'ATC_L1_NAME'] = ATC_L1_NAMES.get(code[0], '')
    if len(code) >= 3:
        atc_df.at[idx, 'ATC_L2_CODE'] = code[:3]
        atc_df.at[idx, 'ATC_L2_NAME'] = best['class_name']
    if len(code) >= 4:
        atc_df.at[idx, 'ATC_L3_CODE'] = code[:4]
        atc_df.at[idx, 'ATC_L3_NAME'] = best['class_name']

    new_l2 = atc_df.at[idx, 'ATC_L2_CODE']
    if old_l2 != new_l2:
        rederived += 1

print(f"FIX 1 COMPLETE: Re-prioritized {rederived} drugs to systemic use")

# Verify key drugs
print(f"\nVerification — key drugs after priority fix:")
verify_drugs = ['ASPIRIN', 'IBUPROFEN', 'DEXAMETHASONE', 'HYDROCORTISONE',
                'PREDNISOLONE', 'BETAMETHASONE', 'LIDOCAINE', 'EPINEPHRINE',
                'METRONIDAZOLE', 'AMPHOTERICIN B', 'ADVIL', 'DECADRON']
for drug in verify_drugs:
    row = atc_df[atc_df['DRUG'] == drug]
    if len(row) > 0:
        r = row.iloc[0]
        print(f"  {drug:30s} -> {r['ATC_L2_CODE']} ({r['ATC_L2_NAME']})")

FIX 1 COMPLETE: Re-prioritized 61 drugs to systemic use

Verification — key drugs after priority fix:
  ASPIRIN                        -> B01 (Platelet aggregation inhibitors excl. heparin)
  IBUPROFEN                      -> N02 (Opioids in combination with non-opioid analgesics)
  DEXAMETHASONE                  -> H02 (Glucocorticoids)
  HYDROCORTISONE                 -> H02 (Glucocorticoids)
  PREDNISOLONE                   -> H02 (Glucocorticoids)
  BETAMETHASONE                  -> H02 (Glucocorticoids)
  LIDOCAINE                      -> C01 (Antiarrhythmics, class Ib)
  EPINEPHRINE                    -> C01 (Adrenergic and dopaminergic agents)
  METRONIDAZOLE                  -> J01 (Combinations of antibacterials)
  AMPHOTERICIN B                 -> J02 (Antibiotics)
  DECADRON                       -> H02 (Glucocorticoids)


In [24]:
# =============================================================================
# FIX 2: CLASSIFY UNCLASSIFIED DRUGS
# =============================================================================
# Three strategies:
#   A) Manual mapping for known brand names and international variants
#   B) Try stripping dosage/formulation info and re-query
#   C) Try backslash-separated combinations

# ---- STRATEGY A: Manual mapping for known drugs ----
# These are drugs we can confidently classify by inspection
MANUAL_OVERRIDES = {
    # International name variants (Spanish/Portuguese/Latin)
    'AMITRIPTILINA':      ('N06AA', 'Non-selective monoamine reuptake inhibitors'),
    'AMLODIPINO':         ('C08CA', 'Dihydropyridine derivatives'),
    'ATORVASTATINA':      ('C10AA', 'HMG CoA reductase inhibitors'),
    'AZILSARTAN KAMEDOXOMIL': ('C09CA', 'Angiotensin II receptor blockers'),
    'BEDAQUILINA':        ('J04AK', 'Other drugs for treatment of tuberculosis'),
    'BROMAZEPAMUM':       ('N05BA', 'Benzodiazepine derivatives'),
    'CALCIO':             ('A12AA', 'Calcium'),
    'CASPOFUNGINA':       ('J02AX', 'Other systemic antifungals'),
    'CEFEPIMA':           ('J01DE', 'Fourth-generation cephalosporins'),
    'CETIRIZINA':         ('R06AE', 'Piperazine derivatives'),
    'CLEMASTINA':         ('R06AA', 'Aminoalkyl ethers'),
    'CLOPIDOGRELUM':      ('B01AC', 'Platelet aggregation inhibitors'),
    'CLORHEXIDINA':       ('D08AC', 'Biguanides and amidines'),
    'DICLOFENACO':        ('M01AB', 'Acetic acid derivatives'),
    'DILTIAZEMUM':        ('C08DB', 'Benzothiazepine derivatives'),
    'DOXORUBICINA':       ('L01DB', 'Anthracyclines'),
    'EPARINA':            ('B01AB', 'Heparin group'),
    'FLUOXETINUM':        ('N06AB', 'Selective serotonin reuptake inhibitors'),
    'HYDROXYZINUM':       ('N05BB', 'Diphenylmethane derivatives'),
    'IDROMORFONE':        ('N02AA', 'Natural opium alkaloids'),
    'IVABRADINA':         ('C01EB', 'Other cardiac preparations'),
    'LOPERAMIDA':         ('A07DA', 'Antipropulsives'),
    'METFORMINUM':        ('A10BA', 'Biguanides'),
    'METOCLOPRAMIDA':     ('A03FA', 'Propulsives'),
    'MICAFUNGINA':        ('J02AX', 'Other systemic antifungals'),
    'NALOXONA':           ('V03AB', 'Antidotes'),
    'PAROXETINUM':        ('N06AB', 'Selective serotonin reuptake inhibitors'),
    'PROMETAZINA':        ('R06AD', 'Phenothiazine derivatives'),
    'PROPANOLOL':         ('C07AA', 'Beta blocking agents, non-selective'),
    'QUETIAPINA':         ('N05AH', 'Diazepines, oxazepines, thiazepines'),
    'ROSUVASTATINA':      ('C10AA', 'HMG CoA reductase inhibitors'),
    'SERTRALINA':         ('N06AB', 'Selective serotonin reuptake inhibitors'),
    'TAMSULOSINA':        ('G04CA', 'Alpha-adrenoreceptor antagonists'),
    'TRAZODONA':          ('N06AX', 'Other antidepressants'),
    'VENLAFAXINA':        ('N06AX', 'Other antidepressants'),
    'ACIDO ALENDRONICO':  ('M05BA', 'Bisphosphonates'),
    'ACIDO FOLINICO':     ('V03AF', 'Detoxifying agents for antineoplastic treatment'),

    # Brand names -> generic ATC
    'ABILIFY MAINTENA':   ('N05AX', 'Other antipsychotics'),         # aripiprazole
    'ACUPAN':             ('N02BG', 'Other analgesics and antipyretics'),  # nefopam
    'ADVAIR HFA':         ('R03AK', 'Adrenergics with corticosteroids'),   # fluticasone/salmeterol
    'ANORO ELLIPTA':      ('R03AL', 'Adrenergics with anticholinergics'),  # umeclidinium/vilanterol
    'CELLCEPT':           ('L04AA', 'Selective immunosuppressants'),  # mycophenolate
    'DOLIPRANE':          ('N02BE', 'Anilides'),                     # paracetamol (French brand)
    'ELOXATIN':           ('L01XA', 'Platinum compounds'),           # oxaliplatin
    'ERELZI':             ('L04AB', 'TNF-alpha inhibitors'),         # etanercept biosimilar
    'KARDEGIC':           ('B01AC', 'Platelet aggregation inhibitors'),  # aspirin (French brand)
    'NOVORAPID':          ('A10AB', 'Insulins, fast-acting'),        # insulin aspart
    'PAXLOVID':           ('J05AE', 'Protease inhibitors'),          # nirmatrelvir/ritonavir
    'SKENAN':             ('N02AA', 'Natural opium alkaloids'),      # morphine (French brand)
    'TRELEGY ELLIPTA':    ('R03AL', 'Adrenergics with anticholinergics'),
    'ZENHALE':            ('R03AK', 'Adrenergics with corticosteroids'),
    'AMEVIVE':            ('L04AA', 'Selective immunosuppressants'),  # alefacept
    'COCODAMOL':          ('N02AJ', 'Opioids with non-opioid analgesics'),  # paracetamol/codeine
    'POLARAMINE':         ('R06AB', 'Substituted alkylamines'),      # dexchlorpheniramine
    'SPIROCTAN':          ('C03DA', 'Aldosterone antagonists'),      # spironolactone
    'MOVICOL':            ('A06AD', 'Osmotically acting laxatives'), # macrogol
    'MOVICOLON':          ('A06AD', 'Osmotically acting laxatives'),
    'ALIGN':              ('A07FA', 'Antidiarrheal micro-organisms'),  # probiotic
    'DECADRON':           ('H02AB', 'Glucocorticoids'),              # dexamethasone brand

    # Specific substances
    'ALCOHOL':            ('V03AZ', 'Ethanol'),
    'ALENDRONATE SODIUM': ('M05BA', 'Bisphosphonates'),
    'ALPHA TOCOPHEROL':   ('A11HA', 'Other plain vitamin preparations'),
    'ALBUMIN HUMAN':      ('B05AA', 'Blood substitutes and plasma protein fractions'),
    'BOTULINUM TOXIN TYPE A': ('M03AX', 'Other muscle relaxants'),
    'CANNABIS SATIVA FLOWERING TOP':  ('N02BG', 'Other analgesics'),
    'CANNABIS SATIVA SUBSP. INDICA TOP': ('N02BG', 'Other analgesics'),
    'CLAVULANIC ACID':    ('J01CR', 'Combinations of penicillins'),
    'CYANOCOBALAMIN':     ('B03BA', 'Vitamin B12'),
    'DEXTROSE':           ('B05BA', 'Solutions for parenteral nutrition'),
    'DIVALPROEX SODIUM':  ('N03AG', 'Fatty acid derivatives'),
    'G CSF':              ('L03AA', 'Colony stimulating factors'),
    'HUMAN IMMUNOGLOBULIN G': ('J06BA', 'Immunoglobulins, normal human'),
    'INSULIN':            ('A10AB', 'Insulins'),
    'INSULIN ASPART':     ('A10AB', 'Insulins, fast-acting'),
    'LITHIUM CARBONATE':  ('N05AN', 'Lithium'),
    'MAGNESIUM':          ('A12CC', 'Magnesium'),
    'MYCOPHENOLATE MOFETIL': ('L04AA', 'Selective immunosuppressants'),
    'POLYETHYLENE GLYCOL': ('A06AD', 'Osmotically acting laxatives'),
    'REBAMIPIDE':         ('A02BX', 'Other drugs for peptic ulcer'),
    'VALPROATE SODIUM':   ('N03AG', 'Fatty acid derivatives'),
    'VASOPRESSIN':        ('H01BA', 'Vasopressin and analogues'),
    'AGOMELATINE':        ('N06AX', 'Other antidepressants'),
    'AZOSEMIDE':          ('C03CA', 'Sulfonamides, plain'),
    'COVID-19 VACCINE MRNA': ('J07BX', 'Other viral vaccines'),
    'ANTI-THYMOCYTE GLOBULIN': ('L04AA', 'Selective immunosuppressants'),
    'ALPHA1-PROTEINASE INHIBITOR HUMAN': ('B02AB', 'Proteinase inhibitors'),
    'BENZOYLECGONINE':    ('N01BX', 'Other local anesthetics'),  # cocaine metabolite
    'ORIDONIN':           ('L01XX', 'Other antineoplastic agents'),
    'TPN':                ('B05BA', 'Solutions for parenteral nutrition'),
    'LT4':                ('H03AA', 'Thyroid hormones'),  # levothyroxine

    # New brand names
    'BACTRIM DS':         ('J01EE', 'Combinations of sulfonamides and trimethoprim'),
    'BREO ELLIPTA':       ('R03AK', 'Adrenergics with corticosteroids'),
    'CALONAL':            ('N02BE', 'Anilides'),                      # paracetamol (Japanese)
    'ENVARSUS XR':        ('L04AD', 'Calcineurin inhibitors'),        # tacrolimus
    'FLONASE ALLERGY RELIEF': ('R01AD', 'Corticosteroids'),
    'LANTUS SOLOSTAR':    ('A10AE', 'Insulins, long-acting'),
    'MIRALAX':            ('A06AD', 'Osmotically acting laxatives'),
    'PULMICORT TURBUHALER': ('R03BA', 'Glucocorticoids'),
    'RIVOTRIL':           ('N03AE', 'Benzodiazepine derivatives'),    # clonazepam
    'SPIRIVA RESPIMAT':   ('R03BB', 'Anticholinergics'),
    'TRIKAFTA':           ('R07AX', 'Other respiratory system products'),
    'TYVASO DPI':         ('B01AC', 'Platelet aggregation inhibitors'),  # treprostinil
    'TYLENOL EXTRA STRENGTH': ('N02BE', 'Anilides'),
    'VENTOLIN HFA':       ('R03AC', 'Selective beta-2-adrenoreceptor agonists'),
    'VOLTARENE':          ('M01AB', 'Acetic acid derivatives'),       # diclofenac (French)
    'ZOFRAN':             ('A04AA', 'Serotonin antagonists'),         # ondansetron

    # International variants
    'CICLOBENZAPRINA':    ('M03BX', 'Other centrally acting agents'),
    'DONEPEZILO':         ('N06DA', 'Anticholinesterases'),
    'DULOXETINA':         ('N06AX', 'Other antidepressants'),
    'EPLERENONA':         ('C03DA', 'Aldosterone antagonists'),
    'FLUTICASONA':        ('R03BA', 'Glucocorticoids'),
    'GLUCOSAMINA':        ('M01AX', 'Other anti-inflammatory agents'),
    'KORTISON':           ('H02AB', 'Glucocorticoids'),
    'LABETOLOL':          ('C07AG', 'Alpha and beta blocking agents'),
    'LERCANIDIPINO':      ('C08CA', 'Dihydropyridine derivatives'),
    'MELATONINA':         ('N05CH', 'Melatonin receptor agonists'),
    'PIRIDOSTIGMINA':     ('N07AA', 'Anticholinesterases'),
    'RANITIDINA':         ('A02BA', 'H2-receptor antagonists'),
    'SILDENAFILO':        ('G04BE', 'Drugs used in erectile dysfunction'),
    'VALACICLOVIR':       ('J05AB', 'Nucleosides and nucleotides'),
    'ISOSORBIDI DINITRAS': ('C01DA', 'Organic nitrates'),

    # Substances / supplements
    'CALCIUM':            ('A12AA', 'Calcium'),
    'FERRUM':             ('B03AA', 'Iron bivalent, oral preparations'),
    'FOLINIC ACID':       ('V03AF', 'Detoxifying agents for antineoplastic treatment'),
    'GOLD':               ('M01CB', 'Gold preparations'),
    'LECITHIN':           ('A05BA', 'Liver therapy, lipotropics'),
    'RISEDRONATE SODIUM': ('M05BA', 'Bisphosphonates'),
    'SENNOSIDES':         ('A06AB', 'Contact laxatives'),
    'VITAMIN B COMPLEX':  ('A11EA', 'Vitamin B-complex'),
    'VITAMIN D NOS':      ('A11CC', 'Vitamin D and analogues'),
    'VITAMIN D3':         ('A11CC', 'Vitamin D and analogues'),
    'VITAMINS':           ('A11BA', 'Multivitamins'),
    'ISMN':               ('C01DA', 'Organic nitrates'),              # isosorbide mononitrate

    # Other
    'POLOXAMER':          ('A06AX', 'Other drugs for constipation'),
    'POLYETHYLENE GLYCOL 3350': ('A06AD', 'Osmotically acting laxatives'),
    'POLYETHYLENE GLYCOL 4000': ('A06AD', 'Osmotically acting laxatives'),
}

# ---- STRATEGY B: Try backslash-separated combinations ----
BACKSLASH_COMBOS = {}
for drug in atc_df[atc_df['ATC_L1_CODE'] == '']['DRUG'].tolist():
    if '\\' in drug:
        parts = drug.split('\\')
        # Use the first component for classification
        first = parts[0].strip()
        entries = get_atc_codes(first, cache)
        if entries:
            best = pick_best_atc(entries)
            BACKSLASH_COMBOS[drug] = (best['atc_code'], best['class_name'])

# ---- APPLY ALL FIXES ----
fixed_manual = 0
fixed_backslash = 0

for idx, row in atc_df[atc_df['ATC_L1_CODE'] == ''].iterrows():
    drug = row['DRUG']

    override = None
    if drug in MANUAL_OVERRIDES:
        code, name = MANUAL_OVERRIDES[drug]
        override = (code, name)
        fixed_manual += 1
    elif drug in BACKSLASH_COMBOS:
        code, name = BACKSLASH_COMBOS[drug]
        override = (code, name)
        fixed_backslash += 1

    if override:
        code, name = override
        if len(code) >= 1:
            atc_df.at[idx, 'ATC_L1_CODE'] = code[0]
            atc_df.at[idx, 'ATC_L1_NAME'] = ATC_L1_NAMES.get(code[0], '')
        if len(code) >= 3:
            atc_df.at[idx, 'ATC_L2_CODE'] = code[:3]
            atc_df.at[idx, 'ATC_L2_NAME'] = name
        if len(code) >= 4:
            atc_df.at[idx, 'ATC_L3_CODE'] = code[:4]
            atc_df.at[idx, 'ATC_L3_NAME'] = name
        atc_df.at[idx, 'ATC_CODE_FULL'] = code

save_cache(cache)

still_unclassified = atc_df[atc_df['ATC_L1_CODE'] == '']['DRUG'].tolist()

print(f"FIX 2 COMPLETE")
print(f"  Manual overrides applied:  {fixed_manual}")
print(f"  Backslash combos fixed:    {fixed_backslash}")
print(f"  Still unclassified:        {len(still_unclassified)}")
print(f"  Coverage rate:             {(atc_df['ATC_L1_CODE'] != '').sum() / len(atc_df) * 100:.1f}%")

if still_unclassified:
    print(f"\nRemaining unclassified:")
    for d in sorted(still_unclassified):
        print(f"  {d}")

FIX 2 COMPLETE
  Manual overrides applied:  87
  Backslash combos fixed:    5
  Still unclassified:        25
  Coverage rate:             96.3%

Remaining unclassified:
  (-)-ARTERENOL
  AZTREONAM LYSINE
  BMS-830216
  CAVIAR, UNSPECIFIED
  DARATUMUMAB/HYALURONIDASE-FIHJ
  DELFLEX WITH DEXTROSE 15% LOW MAGNESIUM LOW CALCIUM
  DELFLEX WITH DEXTROSE 25% LOW MAGNESIUM LOW CALCIUM
  DELFLEX WITH DEXTROSE 425% LOW MAGNESIUM LOW CALCIUM
  DEVICE
  DIANEAL PD-2 PERITONEAL DIALYSIS SOLUTION WITH DEXTROSE
  DIANEAL PD-2 WITH DEXTROSE
  DOXAZOSINA
  FISHOIL
  FOSCARBIDOPA/FOSLEVODOPA
  GIMERACIL\OTERACIL\TEGAFUR
  LIPOGEN
  MENTHOL/MINERAL OIL/VITAMIN A
  MINOCICLINA
  PERPRAZOLE
  REGUNEAL LCA
  RETINOL
  STERILE WATER
  TIRAGOLUMAB
  URSADIOL 300 MG ORAL CAPSULE [URSODIOL]
  ZINCUM METALLICUM


## Step 7: Map to High-Risk Drug Classes

Research plan specified these priority classes:
- Anticoagulants / Antithrombotics (ATC B01)
- Antineoplastics (ATC L01)
- Immunosuppressants (ATC L04)
- Opioids (ATC N02A)
- Narrow Therapeutic Index drugs (various ATC codes)
- Anti-inflammatory / Analgesics (ATC M01, N02)

We map ATC codes to these research-relevant categories.


In [25]:
def atc_to_risk_class(row):
    """Map ATC codes to clinically meaningful risk categories for the paper."""
    l1 = row['ATC_L1_CODE']
    l2 = row['ATC_L2_CODE']
    drug = row['DRUG'].upper()

    # CHECK NTI FIRST — cross-cutting category that overrides ATC grouping
    nti_drugs = ['DIGOXIN', 'LITHIUM', 'PHENYTOIN', 'CARBAMAZEPINE',
                 'THEOPHYLLINE', 'VANCOMYCIN', 'GENTAMICIN', 'WARFARIN',
                 'CYCLOSPORINE', 'TACROLIMUS', 'VALPROIC', 'DIVALPROEX',
                 'VALPROATE']
    for nti in nti_drugs:
        if nti in drug:
            return 'Narrow Therapeutic Index'

    # Specific ATC Level 2 mappings
    if l2.startswith('B01'):
        return 'Antithrombotic'
    if l2.startswith('L01'):
        return 'Antineoplastic'
    if l2.startswith('L04'):
        return 'Immunosuppressant'
    if l2.startswith('N02'):
        return 'Analgesic/Opioid'
    if l2.startswith('N01'):
        return 'Anesthetic'
    if l2.startswith('M01'):
        return 'Anti-inflammatory (NSAID)'
    if l2.startswith('J01'):
        return 'Antibacterial'
    if l2.startswith('J05'):
        return 'Antiviral'
    if l2.startswith('C09'):
        return 'RAAS Inhibitor'
    if l2.startswith('C07'):
        return 'Beta Blocker'
    if l2.startswith('C10'):
        return 'Lipid Modifying'
    if l2.startswith('A10'):
        return 'Antidiabetic'
    if l2.startswith('N05'):
        return 'Psycholeptic'
    if l2.startswith('N06'):
        return 'Psychoanaleptic'
    if l2.startswith('N03'):
        return 'Antiepileptic'
    if l2.startswith('H02'):
        return 'Corticosteroid'
    if l2.startswith('R03'):
        return 'Anti-asthmatic'

    # Fall back to Level 1
    l1_fallback = {
        'A': 'Alimentary/Metabolism',
        'B': 'Blood (Other)',
        'C': 'Cardiovascular (Other)',
        'D': 'Dermatological',
        'G': 'Genitourinary',
        'H': 'Hormonal',
        'J': 'Anti-infective (Other)',
        'L': 'Antineoplastic/Immuno (Other)',
        'M': 'Musculoskeletal (Other)',
        'N': 'Nervous System (Other)',
        'P': 'Antiparasitic',
        'R': 'Respiratory (Other)',
        'S': 'Sensory',
        'V': 'Various'
    }
    if l1 in l1_fallback:
        return l1_fallback[l1]

    return 'Unclassified'

atc_df['RISK_CLASS'] = atc_df.apply(atc_to_risk_class, axis=1)

print("HIGH-RISK CLASS DISTRIBUTION")
print("=" * 60)
risk_counts = atc_df['RISK_CLASS'].value_counts()
for cls, count in risk_counts.items():
    marker = " ***" if cls in ['Antithrombotic', 'Antineoplastic', 'Immunosuppressant',
                                'Analgesic/Opioid', 'Narrow Therapeutic Index',
                                'Anti-inflammatory (NSAID)', 'Corticosteroid'] else ""
    print(f"  {cls:40s} {count:4d}{marker}")
print(f"\n*** = priority high-risk classes")

HIGH-RISK CLASS DISTRIBUTION
  Alimentary/Metabolism                      69
  Antineoplastic                             61 ***
  Immunosuppressant                          43 ***
  Cardiovascular (Other)                     43
  Psycholeptic                               35
  Antibacterial                              34
  Antidiabetic                               34
  Analgesic/Opioid                           32 ***
  Anti-asthmatic                             28
  Psychoanaleptic                            27
  Unclassified                               25
  Antithrombotic                             23 ***
  Respiratory (Other)                        22
  Musculoskeletal (Other)                    19
  RAAS Inhibitor                             17
  Anti-inflammatory (NSAID)                  16 ***
  Narrow Therapeutic Index                   16 ***
  Blood (Other)                              14
  Antiepileptic                              13
  Anti-infective (Other)           

## Step 8: Save Mapping File

In [26]:
# ---- SAVE MAPPING ----
atc_df.to_csv("drug_atc_mapping.csv", index=False)
print(f"Saved: drug_atc_mapping.csv ({len(atc_df)} drugs)")

# ---- ALSO SAVE A SUMMARY TABLE FOR THE PAPER ----
summary = atc_df.groupby('RISK_CLASS').agg(
    DRUG_COUNT=('DRUG', 'count'),
    EXAMPLE_DRUGS=('DRUG', lambda x: ', '.join(sorted(x)[:5]))
).sort_values('DRUG_COUNT', ascending=False)

summary.to_csv("atc_class_summary.csv")
print(f"Saved: atc_class_summary.csv ({len(summary)} classes)")

print(f"\n{'=' * 60}")
print("ATC CLASSIFICATION COMPLETE")
print(f"{'=' * 60}")
print(f"\nFiles created:")
print(f"  drug_atc_mapping.csv     — full drug-to-ATC mapping")
print(f"  atc_class_summary.csv    — class distribution summary")
print(f"  atc_mapping_cache.json   — API response cache (for re-runs)")
print(f"\nNext steps:")
print(f"  1. Update DDI_Network_Visualization.ipynb to use drug_atc_mapping.csv")
print(f"     instead of the manual high_risk_classes dictionary")
print(f"  2. Regenerate Gephi node files with ATC-based DRUG_CLASS column")
print(f"  3. Optionally: create class-level aggregated network graph")


Saved: drug_atc_mapping.csv (682 drugs)
Saved: atc_class_summary.csv (33 classes)

ATC CLASSIFICATION COMPLETE

Files created:
  drug_atc_mapping.csv     — full drug-to-ATC mapping
  atc_class_summary.csv    — class distribution summary
  atc_mapping_cache.json   — API response cache (for re-runs)

Next steps:
  1. Update DDI_Network_Visualization.ipynb to use drug_atc_mapping.csv
     instead of the manual high_risk_classes dictionary
  2. Regenerate Gephi node files with ATC-based DRUG_CLASS column
  3. Optionally: create class-level aggregated network graph


In [27]:
import json
cache = json.load(open("atc_mapping_cache.json"))

# Check a few drugs we know should classify well
test_drugs = ['WARFARIN', 'METHOTREXATE', 'ASPIRIN', 'MORPHINE', 'ADALIMUMAB']
for drug in test_drugs:
    if drug in cache:
        print(f"\n{drug}:")
        for entry in cache[drug]:
            print(f"  {entry['atc_code']:10s} (level {entry['atc_level']}) — {entry['class_name']}")
    else:
        print(f"\n{drug}: NOT IN CACHE")


WARFARIN:
  B01AA      (level 5) — Vitamin K antagonists

METHOTREXATE:
  L01BA      (level 5) — Folic acid analogues
  L04AX      (level 5) — Other immunosuppressants

ASPIRIN:
  A01AD      (level 5) — Other agents for local oral treatment
  B01AC      (level 5) — Platelet aggregation inhibitors excl. heparin
  N02BA      (level 5) — Salicylic acid and derivatives
  N02AJ      (level 5) — Opioids in combination with non-opioid analgesics

MORPHINE:
  N02AA      (level 5) — Natural opium alkaloids

ADALIMUMAB:
  L04AB      (level 5) — Tumor necrosis factor alpha (TNF-alpha) inhibitors


In [28]:
multi = atc_df[atc_df['NUM_ATC_CLASSES'] > 1]
print(f"Drugs with multiple ATC codes: {len(multi)}")
print(f"\nTop 20 by number of classes:")
for _, row in multi.nlargest(20, 'NUM_ATC_CLASSES').iterrows():
    entries = results.get(row['DRUG'], [])
    codes = [e['atc_code'] for e in entries]
    print(f"  {row['DRUG']:35s} -> {', '.join(codes)}")

Drugs with multiple ATC codes: 178

Top 20 by number of classes:
  BETAMETHASONE                       -> A07EA, C05AA, D07AC, D07XC, H02AB, R01AD, R03BA, S01BA, S01CB, S02BA, S03BA
  DECADRON                            -> A01AC, C05AA, D07AB, D07XB, D10AA, H02AB, R01AD, S01BA, S01CB, S02BA, S03BA
  DEXAMETHASONE                       -> A01AC, C05AA, D07AB, D07XB, D10AA, H02AB, R01AD, S01BA, S01CB, S02BA, S03BA
  HYDROCORTISONE                      -> D07AB, D07AB, D07AC, A01AC, A07EA, C05AA, D07AA, D07XA, H02AB, S01BA, S01CB, S02BA
  PREDNISOLONE                        -> A01AC, A07EA, C05AA, D07AA, D07XA, H02AB, R01AD, S01BA, S01CB, S02BA, S03BA
  HYDROCORTISONE ACETATE              -> A01AC, A07EA, C05AA, D07AA, D07XA, H02AB, S01BA, S01CB, S02BA
  CHLORHEXIDINE GLUCONATE             -> A01AB, B05CA, D08AC, D09AA, R02AA, S01AX, S02AA, S03AA
  TRIAMCINOLONE ACETONIDE             -> A01AC, C05AA, D07AB, D07XB, H02AB, R01AD, R03BA, S01BA
  BREZTRI                             -> D11AA, 